# Fleet Performance Analytics — Data Generation

**Purpose:** Generate a reproducible synthetic dataset for a Zimbabwean trucking operation with realistic relationships that can support valid statistical analysis and predictive modelling.

### What fixes
- Fuel consumption is generated from **pre-trip variables**; `fuel_efficiency_km_per_l` is not used to generate or predict the target.
- Delay events are driven by route risk, border crossings, road conditions, weather, loading time, truck age and driver factors instead of a purely random 20% flag.
- Maintenance frequency depends on truck age, mileage and utilization.
- Maintenance cost depends on age, mileage, category and severity.
- Trip revenue and total operating costs allow both profitable and loss-making trips.
- The dataset retains the original core tables and adds transport-specific operational features.

> **Portfolio note:** This is synthetic data for a portfolio project. It is not real company data.

In [ ]:
# Cell 1 — Imports, seed and paths
from pathlib import Path
import random
import numpy as np
import pandas as pd
from faker import Faker

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
Faker.seed(SEED)
fake = Faker()

REFERENCE_DATE = pd.Timestamp('2026-09-22')
START_DATE = REFERENCE_DATE - pd.Timedelta(days=365)

RAW_DATA_PATH = Path('../data/raw')
RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)

NUM_TRUCKS = 35
NUM_DRIVERS = 45
NUM_TRIPS = 1500
NUM_MAINTENANCE = 350
NUM_FUEL_PURCHASES = 500

print('Seed:', SEED)
print('Reference date:', REFERENCE_DATE.date())
print('Output:', RAW_DATA_PATH.resolve())

In [ ]:
# Cell 2 — Fleet configuration and route definitions
CITIES = ['Harare','Bulawayo','Mutare','Gweru','Masvingo','Beitbridge','Chirundu','Kwekwe','Kadoma','Chinhoyi']

TRUCK_MODELS = {
    'Scania R450': {'year_range': (2018, 2023), 'base_consumption': 32, 'capacity': 25},
    'Volvo FH500': {'year_range': (2017, 2022), 'base_consumption': 30, 'capacity': 28},
    'MAN TGX': {'year_range': (2016, 2021), 'base_consumption': 34, 'capacity': 24},
    'Mercedes Actros': {'year_range': (2018, 2023), 'base_consumption': 28, 'capacity': 26},
    'DAF XF': {'year_range': (2015, 2020), 'base_consumption': 36, 'capacity': 22},
    'Iveco Stralis': {'year_range': (2014, 2019), 'base_consumption': 38, 'capacity': 20},
    'Renault T': {'year_range': (2016, 2022), 'base_consumption': 33, 'capacity': 23},
}

MAINTENANCE_CATEGORIES = {
    'Engine Service': {'avg_cost': 850, 'duration_days': (1,3), 'severity_weight': 1.15},
    'Brake Replacement': {'avg_cost': 450, 'duration_days': (1,2), 'severity_weight': 1.05},
    'Transmission Repair': {'avg_cost': 1200, 'duration_days': (2,5), 'severity_weight': 1.30},
    'Tire Replacement': {'avg_cost': 300, 'duration_days': (1,1), 'severity_weight': 0.95},
    'Oil Change': {'avg_cost': 150, 'duration_days': (1,1), 'severity_weight': 0.80},
    'Electrical Repair': {'avg_cost': 500, 'duration_days': (1,2), 'severity_weight': 1.00},
    'Cooling System': {'avg_cost': 400, 'duration_days': (1,2), 'severity_weight': 1.00},
    'Suspension Repair': {'avg_cost': 700, 'duration_days': (1,3), 'severity_weight': 1.15},
    'Clutch Replacement': {'avg_cost': 950, 'duration_days': (2,4), 'severity_weight': 1.25},
    'Full Service': {'avg_cost': 600, 'duration_days': (1,2), 'severity_weight': 1.05},
}

ROUTE_DISTANCES = {
    ('Harare','Bulawayo'):440, ('Harare','Mutare'):263, ('Harare','Gweru'):275,
    ('Harare','Masvingo'):292, ('Harare','Beitbridge'):582, ('Harare','Chirundu'):335,
    ('Bulawayo','Mutare'):500, ('Bulawayo','Gweru'):165, ('Bulawayo','Masvingo'):260,
    ('Bulawayo','Beitbridge'):270, ('Bulawayo','Chirundu'):775, ('Mutare','Gweru'):365,
    ('Mutare','Masvingo'):380, ('Mutare','Beitbridge'):630, ('Mutare','Chirundu'):598,
    ('Gweru','Masvingo'):160, ('Gweru','Beitbridge'):375, ('Gweru','Chirundu'):610,
    ('Masvingo','Beitbridge'):215, ('Masvingo','Chirundu'):627, ('Beitbridge','Chirundu'):830,
}

BORDER_POSTS = {'Beitbridge':'Beitbridge Border Post', 'Chirundu':'Chirundu Border Post'}
ROUTE_RISK = {
    ('Harare','Beitbridge'):0.65, ('Mutare','Beitbridge'):0.70, ('Bulawayo','Beitbridge'):0.55,
    ('Masvingo','Beitbridge'):0.58, ('Gweru','Beitbridge'):0.52, ('Harare','Chirundu'):0.45,
    ('Bulawayo','Chirundu'):0.72, ('Mutare','Chirundu'):0.68, ('Gweru','Chirundu'):0.62,
    ('Masvingo','Chirundu'):0.64, ('Beitbridge','Chirundu'):0.78,
}

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [ ]:
# Cell 3 — Generate trucks and drivers
trucks=[]
for i in range(1, NUM_TRUCKS+1):
    model=random.choice(list(TRUCK_MODELS))
    info=TRUCK_MODELS[model]
    year=random.randint(*info['year_range'])
    age=REFERENCE_DATE.year-year
    annual_km=rng.normal(85000,12000)
    mileage=max(20000, int(age*annual_km + rng.normal(0,15000)))
    capacity=round(max(15, info['capacity']+rng.normal(0,1.2)),1)
    registration=f"{''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ',k=3))} {''.join(random.choices('0123456789',k=4))}"
    trucks.append({'truck_id':f'TRK{i:03d}','model':model,'year':year,'age':age,
                   'purchase_date':pd.Timestamp(year,random.randint(1,12),random.randint(1,28)).strftime('%Y-%m-%d'),
                   'mileage_km':mileage,'capacity_tons':capacity,'registration':registration,
                   'base_consumption_l_per_100km':info['base_consumption']})
trucks_df=pd.DataFrame(trucks)

drivers=[]
for i in range(1, NUM_DRIVERS+1):
    exp=random.randint(2,30)
    skill=float(np.clip(rng.normal(0.75,0.12),0.45,0.98))
    rating=round(np.clip(2.8 + skill*2.0 + rng.normal(0,0.18),2.5,5.0),1)
    first,last=fake.first_name(),fake.last_name()
    drivers.append({'driver_id':f'DRV{i:03d}','first_name':first,'last_name':last,'full_name':f'{first} {last}',
                    'license_type':random.choices(['Class 1','Class 2','Class 3','Class 4'],weights=[.30,.30,.25,.15])[0],
                    'experience_years':exp,'hire_date':(REFERENCE_DATE-pd.Timedelta(days=exp*365+random.randint(0,365))).strftime('%Y-%m-%d'),
                    'contact':f"+263 7{random.randint(1,9)} {random.randint(100,999)} {random.randint(1000,9999)}",
                    'rating':rating,'status':random.choices(['Available','On Trip','On Leave'],weights=[.6,.3,.1])[0],
                    'driver_skill_score':round(skill*100,1)})
drivers_df=pd.DataFrame(drivers)

truck_dict=trucks_df.set_index('truck_id').to_dict('index')
driver_dict=drivers_df.set_index('driver_id').to_dict('index')
print('Trucks:', len(trucks_df), '| Drivers:', len(drivers_df))

In [ ]:
# Cell 4 — Generate trips with realistic operational relationships
trips=[]
for i in range(1, NUM_TRIPS+1):
    (origin,destination),distance=random.choice(list(ROUTE_DISTANCES.items()))
    truck_id=random.choice(trucks_df.truck_id.tolist())
    driver_id=random.choice(drivers_df.driver_id.tolist())
    truck=truck_dict[truck_id]; driver=driver_dict[driver_id]
    trip_date=START_DATE + pd.Timedelta(days=random.randint(0,365))
    capacity=truck['capacity_tons']
    cargo=round(float(np.clip(rng.normal(capacity*0.70, capacity*0.16),3,capacity*0.95)),1)
    utilization=cargo/capacity

    route_risk=ROUTE_RISK.get((origin,destination),0.25)
    border_flag=int(origin in BORDER_POSTS or destination in BORDER_POSTS)
    border_name=BORDER_POSTS.get(destination,BORDER_POSTS.get(origin,'None'))
    rainy_season=trip_date.month in [11,12,1,2,3]
    weather=np.random.choice(['Clear','Cloudy','Rain'],p=[.60,.28,.12])
    weather_score={'Clear':0.0,'Cloudy':0.25,'Rain':0.65}[weather]
    road=np.random.choice(['Good','Fair','Poor'],p=[.45,.40,.15])
    road_score={'Good':0.0,'Fair':0.35,'Poor':0.80}[road]
    if rainy_season and road=='Poor': road_score=min(1.0,road_score+0.12)
    loading_hours=round(max(1.0,rng.normal(2.2+utilization*2.2,0.55)),1)

    # Target: fuel consumed. Only pre-trip factors are used here.
    fuel_random=max(0.88,min(1.12,rng.normal(1,0.045)))
    fuel_l=(distance/100)*truck['base_consumption_l_per_100km']*(1+truck['age']*0.006)*(1+max(0,cargo-10)*0.009)*(1+0.10*road_score+0.06*weather_score)*fuel_random
    fuel_l=round(max(20,fuel_l),2)
    fuel_price=round(float(np.clip(rng.normal(1.72,0.10),1.45,2.05)),2)
    fuel_cost=round(fuel_l*fuel_price,2)

    avg_speed=max(38,68-utilization*11-road_score*8-weather_score*5+rng.normal(0,3))
    planned_duration=max(1,distance/avg_speed)
    delay_logit=(-5.30+2.20*route_risk+1.40*border_flag+1.80*road_score+1.50*weather_score+
                 2.00*max(0,utilization-0.65)+0.05*truck['age']+0.035*loading_hours+0.0015*distance)
    delay_prob=float(sigmoid(delay_logit))
    has_delay=int(rng.random()<delay_prob)
    delay_minutes=int(rng.integers(25,181)) if has_delay else 0
    duration=round(max(1,planned_duration+loading_hours+delay_minutes/60+rng.normal(0,0.15)),1)

    # Profit uses total operating cost, allowing a realistic loss-making class.
    revenue=round((1.22*distance+0.012*distance*cargo)*rng.uniform(0.82,1.12),2)
    driver_allowance=round(max(25,0.22*distance+rng.normal(15,5)),2)
    tolls=round((rng.uniform(20,80) if border_flag else rng.uniform(0,30)),2)
    other_operating=round(0.18*distance*rng.uniform(0.9,1.12),2)
    maintenance_allocation=round(0.05*distance*(1+truck['age']*0.03),2)
    operating_cost=round(fuel_cost+driver_allowance+tolls+other_operating+maintenance_allocation,2)
    profit=round(revenue-operating_cost,2)

    trips.append({'trip_id':f'TRP{i:05d}','truck_id':truck_id,'driver_id':driver_id,'origin':origin,'destination':destination,
                  'route':f'{origin} → {destination}','distance_km':distance,'trip_date':trip_date.strftime('%Y-%m-%d'),
                  'cargo_weight_tons':cargo,'cargo_utilization_pct':round(utilization*100,1),
                  'fuel_consumed_litres':fuel_l,'fuel_price_per_litre':fuel_price,'fuel_cost':fuel_cost,
                  'revenue':revenue,'driver_allowance':driver_allowance,'tolls':tolls,'other_operating_cost':other_operating,
                  'maintenance_allocation':maintenance_allocation,'operating_cost':operating_cost,'profit':profit,
                  'planned_duration_hours':round(planned_duration+loading_hours,1),'duration_hours':duration,'loading_time_hours':loading_hours,
                  'has_delay':bool(has_delay),'delay_minutes':delay_minutes,'road_condition':road,'road_condition_score':road_score,
                  'weather_condition':weather,'weather_risk_score':weather_score,'route_risk_score':route_risk,
                  'border_crossing_flag':border_flag,'border_crossing':border_name})
trips_df=pd.DataFrame(trips)
print('Trips:', len(trips_df), '| Delay rate:', f"{trips_df.has_delay.mean()*100:.1f}%", '| Loss rate:', f"{(trips_df.profit<=0).mean()*100:.1f}%")

In [ ]:
# Cell 5 — Generate maintenance records with age/mileage effects
trip_counts=trips_df.truck_id.value_counts().reindex(trucks_df.truck_id).fillna(0)
risk=(trucks_df.set_index('truck_id')['age']*0.9 + trucks_df.set_index('truck_id')['mileage_km']/100000*0.55 + trip_counts/100*0.5 + 2)
weights=(risk/risk.sum()).values
selected=rng.choice(trucks_df.truck_id.values,size=NUM_MAINTENANCE,p=weights)

categories=list(MAINTENANCE_CATEGORIES)
cat_weights=[.12,.12,.05,.18,.18,.07,.06,.07,.05,.10]
maintenance=[]
for i,truck_id in enumerate(selected,1):
    truck=truck_dict[truck_id]
    maint_date=START_DATE+pd.Timedelta(days=random.randint(0,365))
    category=random.choices(categories,weights=cat_weights)[0]
    info=MAINTENANCE_CATEGORIES[category]
    severity=float(np.clip(rng.normal(1,0.18),0.65,1.45))
    age_factor=1+truck['age']*0.045
    mileage_factor=1+(truck['mileage_km']/100000)*0.055
    cost=round(max(75,info['avg_cost']*info['severity_weight']*severity*age_factor*mileage_factor*rng.uniform(.92,1.10)),2)
    base_days=random.randint(*info['duration_days'])
    downtime=max(1,int(round(base_days*(0.85+severity*0.35+truck['age']*0.025))))
    mileage_at=int(max(5000,truck['mileage_km']+rng.normal(0,12000)))
    breakdown_event=int(rng.random()<sigmoid(-3.2+truck['age']*0.22+mileage_at/100000*0.45+severity*0.45))
    maintenance.append({'maintenance_id':f'MAINT{i:04d}','truck_id':truck_id,'maintenance_date':maint_date.strftime('%Y-%m-%d'),
                        'category':category,'severity':round(severity,2),'cost':cost,'downtime_days':downtime,
                        'mileage_at_maintenance':mileage_at,'breakdown_event':breakdown_event,
                        'description':f'{category} - {fake.sentence(nb_words=5)}'})
maintenance_df=pd.DataFrame(maintenance)
print('Maintenance records:', len(maintenance_df))

In [ ]:
# Cell 6 — Generate fuel purchase records
fuel=[]
for i in range(1,NUM_FUEL_PURCHASES+1):
    truck_id=random.choice(trucks_df.truck_id.tolist())
    purchase_date=START_DATE+pd.Timedelta(days=random.randint(0,365))
    location=random.choice(CITIES)
    litres=round(float(np.clip(rng.normal(260,85),60,550)),1)
    price=round(float(np.clip(rng.normal(1.72,0.12),1.40,2.15)),2)
    fuel.append({'fuel_id':f'FUEL{i:05d}','truck_id':truck_id,'purchase_date':purchase_date.strftime('%Y-%m-%d'),
                 'location':location,'litres':litres,'price_per_litre':price,'total_cost':round(litres*price,2)})
fuel_df=pd.DataFrame(fuel)
print('Fuel purchases:', len(fuel_df))

In [ ]:
# Cell 7 — Save all raw tables
for name,df in [('trucks',trucks_df),('drivers',drivers_df),('trips',trips_df),('maintenance',maintenance_df),('fuel_purchases',fuel_df)]:
    df.to_csv(RAW_DATA_PATH/f'{name}.csv',index=False)
    print(f'✓ {name}.csv -> {len(df):,} rows')

In [ ]:
# Cell 8 — Validation: class balance and key relationships
from scipy.stats import pearsonr

maintenance_check=maintenance_df.merge(trucks_df[['truck_id','age','mileage_km']],on='truck_id',how='left')
maintenance_count=maintenance_df.groupby('truck_id').size().rename('maintenance_count')
truck_check=trucks_df.set_index('truck_id').join(maintenance_count).fillna({'maintenance_count':0})

print('TRIP-LEVEL CHECKS')
print(f"  Delay rate: {(trips_df.has_delay.mean()*100):.1f}%")
print(f"  Loss-making trips: {(trips_df.profit<=0).sum():,} ({(trips_df.profit<=0).mean()*100:.1f}%)")
print(f"  Average fuel efficiency: {(trips_df.distance_km/trips_df.fuel_consumed_litres).mean():.2f} km/L")
print(f"  Total revenue: ${trips_df.revenue.sum():,.2f}")
print(f"  Total profit: ${trips_df.profit.sum():,.2f}")

print('MAINTENANCE RELATIONSHIPS')
print(f"  Age vs maintenance count: r={pearsonr(truck_check.age,truck_check.maintenance_count)[0]:.3f}, p={pearsonr(truck_check.age,truck_check.maintenance_count)[1]:.3g}")
print(f"  Mileage vs maintenance count: r={pearsonr(truck_check.mileage_km,truck_check.maintenance_count)[0]:.3f}, p={pearsonr(truck_check.mileage_km,truck_check.maintenance_count)[1]:.3g}")
print(f"  Age vs maintenance cost: r={pearsonr(maintenance_check.age,maintenance_check.cost)[0]:.3f}, p={pearsonr(maintenance_check.age,maintenance_check.cost)[1]:.3g}")
print(f"  Mileage vs maintenance cost: r={pearsonr(maintenance_check.mileage_km,maintenance_check.cost)[0]:.3f}, p={pearsonr(maintenance_check.mileage_km,maintenance_check.cost)[1]:.3g}")

In [ ]:
# Cell 9 — Save validation summary
summary = {
    'reference_date': str(REFERENCE_DATE.date()),
    'trucks': len(trucks_df), 'drivers': len(drivers_df), 'trips': len(trips_df),
    'maintenance_records': len(maintenance_df), 'fuel_purchases': len(fuel_df),
    'delay_rate_pct': round(trips_df.has_delay.mean()*100,2),
    'loss_making_rate_pct': round((trips_df.profit<=0).mean()*100,2),
    'total_revenue': round(trips_df.revenue.sum(),2),
    'total_profit': round(trips_df.profit.sum(),2),
    'average_fuel_efficiency_km_per_l': round((trips_df.distance_km/trips_df.fuel_consumed_litres).mean(),3),
    'age_vs_maintenance_count_r': round(pearsonr(truck_check.age,truck_check.maintenance_count)[0],3),
    'mileage_vs_maintenance_count_r': round(pearsonr(truck_check.mileage_km,truck_check.maintenance_count)[0],3),
    'age_vs_maintenance_cost_r': round(pearsonr(maintenance_check.age,maintenance_check.cost)[0],3),
    'mileage_vs_maintenance_cost_r': round(pearsonr(maintenance_check.mileage_km,maintenance_check.cost)[0],3),
}
pd.Series(summary).to_json(RAW_DATA_PATH/'generation_summary.json',indent=2)
print(pd.Series(summary).to_string())